In [15]:
from pathlib import Path
import sys
root = Path.cwd().resolve().parent

if str(root) not in sys.path:
    sys.path.append(str(root))

In [16]:
from srcs.pipelines.pipeline import PipelineRunner
from utils import cfg

In [17]:
import pandas as pd
import numpy as np
DIR_DATASET_CV = root / "Home Credit Dataset" / "cv"
cv_tables = []

In [18]:
for i in range(1, 6):
    train = pd.read_csv(DIR_DATASET_CV / f"fold_{i}" / f"train_{i}.csv")
    valid = pd.read_csv(DIR_DATASET_CV / f"fold_{i}" / f"valid_{i}.csv")
    common_cols = train.columns.intersection(valid.columns)
    train = train[common_cols]
    valid = valid[common_cols]
    X_train = train.drop(columns=['TARGET']).replace([np.inf, -np.inf], np.nan)
    X_valid= valid.drop(columns=['TARGET']).replace([np.inf, -np.inf], np.nan)
    y_train = train['TARGET']
    y_valid = valid['TARGET']
    tp = (X_train, y_train, X_valid, y_valid)
    cv_tables.append(tp)

In [19]:
train_data = pd.read_csv(cfg.get_processed_train('main_table'))
valid_data = pd.read_csv(cfg.get_processed_valid('main_table'))
test_data = pd.read_csv(cfg.get_processed_test('main_table'))
train_data = train_data.replace([np.inf, -np.inf], np.nan)
test_data = test_data.replace([np.inf, -np.inf], np.nan)
valid_data = valid_data.replace([np.inf, -np.inf], np.nan)

In [20]:
common_cols = train_data.columns.intersection(test_data.columns)
train_data = train_data[common_cols]
valid_data = valid_data[common_cols]
test_data = test_data[common_cols]

In [21]:
X_train = train_data.drop(columns=['TARGET'])
X_test = test_data.drop(columns=['TARGET'])
X_valid = valid_data.drop(columns=['TARGET'])
y_train = train_data['TARGET']
y_test = test_data['TARGET']
y_valid = valid_data['TARGET']

#### Logistic Regression with L2 penalty

In [ ]:
from srcs.pipelines.pipeline import PipelineRunner
from srcs.evaluation import discrimination_report, calibration_report

runner = PipelineRunner(
    mode="valid",
    model_name="lr",
    n_trials=10,
    random_state=42,
    calibration="sigmoid"  # Khuyên dùng "sigmoid" cho Logistic Regression để giữ PD mượt
)

In [22]:
from srcs.evaluation import discrimination_report, calibration_report

In [ ]:
report = runner.run_cv(folds=cv_tables)

In [ ]:
# 2. Chạy Validation Pipeline (huấn luyện, calibration, lưu model và vẽ đồ thị)
valid_report = runner.run_valid(X_train, X_valid, y_train, y_valid)
# 3. Lấy model đã fit và dự đoán xác suất chuẩn hóa
fitted_model = runner._build_model(X_train, valid_report["best_params"])
fitted_model.fit(X_train, y_train)
train_score = runner._predict_score(fitted_model, X_train)
valid_score = runner._predict_score(fitted_model, X_valid)
# 4. Đánh giá chất lượng Calibration (Brier Score & ECE)
calib_metrics = calibration_report(y_valid, valid_score)
print("=== CALIBRATION METRICS ===")
print(f"Brier Score Loss: {calib_metrics['brier_score']:.5f} (Càng gần 0 càng tốt)")
print(f"Expected Calibration Error (ECE): {calib_metrics['expected_calibration_error']:.5f}")
# 5. Đánh giá chi tiết 3 chiến lược Threshold
thresholds_dict = valid_report["thresholds"]
strategies = {
    "1. Giảm nợ xấu (Conservative - 10:1)": thresholds_dict.get("cost_conservative", 0.10),
    "2. Cân bằng rủi ro (Balanced - 5:1)": thresholds_dict.get("cost_balanced", thresholds_dict["youden_j"]),
    "3. Tăng trưởng doanh thu (Growth - 2:1)": thresholds_dict.get("cost_growth", 0.28),
    "Mặc định kỹ thuật (Default 0.50)": 0.50,
}
comparison_rows = []
for strat_name, th in strategies.items():
    rep = discrimination_report(y_valid, valid_score, threshold=th)
    cm = rep["confusion_matrix"]
    comparison_rows.append({
        "Chiến lược": strat_name,
        "Ngưỡng (Threshold)": f"{th:.4f}",
        "ROC-AUC": f"{rep['roc_auc']:.4f}",
        "Recall (Bắt nợ xấu)": f"{rep['recall']*100:.2f}%",
        "Precision (Đúng nợ xấu)": f"{rep['precision']*100:.2f}%",
        "F1-Score": f"{rep['f1']:.4f}",
        "Bỏ lọt nợ xấu (FN)": cm["fn"],
        "Từ chối nhầm (FP)": cm["fp"],
    })
comparison_df = pd.DataFrame(comparison_rows)
print("\n=== BẢNG SO SÁNH HIỆU QUẢ 3 CHIẾN LƯỢC NGƯỠNG ===")

In [ ]:
comparison_df

In [ ]:
test_report = runner.run_test(X_train, X_test, y_train, y_test)
test_report

#### XGBoost

In [ ]:
xgb = PipelineRunner(mode='cv', model_name='xgb', calibration='sigmoid', n_trials=5)

In [ ]:
xgb_tuning = xgb.run_cv(folds=cv_tables)

#### Valid

In [ ]:
# 2. Chạy Validation Pipeline (huấn luyện, calibration, lưu model và vẽ đồ thị)
valid_report = xgb.run_valid(X_train, X_valid, y_train, y_valid)
# 3. Lấy model đã fit và dự đoán xác suất chuẩn hóa
fitted_model = xgb._build_model(X_train, valid_report["best_params"])
fitted_model.fit(X_train, y_train)
train_score = xgb._predict_score(fitted_model, X_train)
valid_score = xgb._predict_score(fitted_model, X_valid)

In [ ]:
# 4. Đánh giá chất lượng Calibration (Brier Score & ECE)
calib_metrics = calibration_report(y_valid, valid_score)
print("=== CALIBRATION METRICS ===")
print(f"Brier Score Loss: {calib_metrics['brier_score']:.5f} (Càng gần 0 càng tốt)")
print(f"Expected Calibration Error (ECE): {calib_metrics['expected_calibration_error']:.5f}")
# 5. Đánh giá chi tiết 3 chiến lược Threshold
thresholds_dict = valid_report["thresholds"]
strategies = {
    "1. Giảm nợ xấu (Conservative - 10:1)": thresholds_dict.get("cost_conservative", 0.10),
    "2. Cân bằng rủi ro (Balanced - 5:1)": thresholds_dict.get("cost_balanced", thresholds_dict["youden_j"]),
    "3. Tăng trưởng doanh thu (Growth - 2:1)": thresholds_dict.get("cost_growth", 0.28),
    "Mặc định kỹ thuật (Default 0.50)": 0.50,
}
comparison_rows = []
for strat_name, th in strategies.items():
    rep = discrimination_report(y_valid, valid_score, threshold=th)
    cm = rep["confusion_matrix"]
    comparison_rows.append({
        "Chiến lược": strat_name,
        "Ngưỡng (Threshold)": f"{th:.4f}",
        "ROC-AUC": f"{rep['roc_auc']:.4f}",
        "Recall (Bắt nợ xấu)": f"{rep['recall']*100:.2f}%",
        "Precision (Đúng nợ xấu)": f"{rep['precision']*100:.2f}%",
        "F1-Score": f"{rep['f1']:.4f}",
        "Bỏ lọt nợ xấu (FN)": cm["fn"],
        "Từ chối nhầm (FP)": cm["fp"],
    })
comparison_df = pd.DataFrame(comparison_rows)
print("\n=== BẢNG SO SÁNH HIỆU QUẢ 3 CHIẾN LƯỢC NGƯỠNG ===")

In [ ]:
comparison_df

In [ ]:
test_report = xgb.run_test(X_train, X_test, y_train, y_test)

In [ ]:
test_report

#### LightGBM

In [25]:
lgbm_runner = PipelineRunner(mode='cv', model_name='lgbm', n_trials=10, calibration='sigmoid')

##### Tuning

In [26]:
lgbm_tuning = lgbm_runner.run_cv(folds=cv_tables)

Trial 1: ROC-AUC=0.85130, params={'selector__top_k': 200, 'model__subsample': 0.8, 'model__num_leaves': 63, 'model__n_estimators': 200, 'model__max_depth': 10, 'model__learning_rate': 0.1}
Trial 2: ROC-AUC=0.85110, params={'selector__top_k': 200, 'model__subsample': 1.0, 'model__num_leaves': 31, 'model__n_estimators': 500, 'model__max_depth': 10, 'model__learning_rate': 0.1}
Trial 3: ROC-AUC=0.84989, params={'selector__top_k': 200, 'model__subsample': 0.8, 'model__num_leaves': 31, 'model__n_estimators': 200, 'model__max_depth': -1, 'model__learning_rate': 0.05}
Trial 4: ROC-AUC=0.85083, params={'selector__top_k': 200, 'model__subsample': 0.8, 'model__num_leaves': 63, 'model__n_estimators': 500, 'model__max_depth': -1, 'model__learning_rate': 0.1}
Trial 5: ROC-AUC=0.85172, params={'selector__top_k': 400, 'model__subsample': 0.8, 'model__num_leaves': 63, 'model__n_estimators': 200, 'model__max_depth': -1, 'model__learning_rate': 0.05}
Trial 6: ROC-AUC=0.85186, params={'selector__top_k': 

##### Valid

In [27]:
valid_lgbm = lgbm_runner.run_valid(X_train, X_valid, y_train, y_valid)

##### Test

In [28]:
test_lgbm = lgbm_runner.run_test(X_train, X_test, y_train, y_test)